In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

In [2]:
BATCH_SIZE = 32
DATA_PATH = 'fifa2021_training.csv'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Usando dispositivo:", DEVICE)

Usando dispositivo: cpu


In [3]:
def load_array(data_arrays, batch_size, is_train=True):
    """Crea un DataLoader desde tensores."""
    dataset = TensorDataset(*data_arrays)
    return DataLoader(dataset, batch_size=batch_size, shuffle=is_train)

In [4]:
def evaluate_loss(net, data_iter, loss_fn):
    """Evalúa la pérdida promedio sobre un iterador."""
    net.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for X_batch, y_batch in data_iter:
            output = net(X_batch)
            loss = loss_fn(output, y_batch)
            total_loss += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)
    return total_loss / total_samples

In [5]:
def accuracy(net, data_iter):
    """Calcula accuracy sobre un iterador completo."""
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in data_iter:
            preds = net(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

In [6]:
def train_epoch_ch3(net, train_iter, loss_fn, optimizer):
    """Entrena una época y devuelve (train_loss, train_accuracy)."""
    net.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_iter:
        optimizer.zero_grad()
        output = net(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y_batch.size(0)
        correct += (output.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total
 

In [7]:
# ─────────────────────────────────────────────────────────
# CARGA Y PREPROCESAMIENTO DEL DATASET
# ─────────────────────────────────────────────────────────

In [8]:
SKILL_COLS = [
    'BallControl', 'Dribbling', 'Marking', 'SlideTackle', 'StandTackle',
    'Aggression', 'Reactions', 'Interceptions', 'Vision', 'Composure',
    'Crossing', 'ShortPass', 'LongPass', 'Acceleration', 'Stamina',
    'Strength', 'Balance', 'SprintSpeed', 'Agility', 'Jumping',
    'Heading', 'ShotPower', 'Finishing', 'LongShots', 'Curve',
    'FKAcc', 'Penalties', 'Volleys', 'GKDiving', 'GKHandling',
    'GKKicking', 'GKReflexes'
]

In [9]:
BASE_COLS = ['Height', 'Weight', 'Age']

In [10]:
def load_and_preprocess(filepath, test_size=0.3, random_state=42):
    """
    Carga el CSV, aplica preprocesamiento y devuelve tensores.
    test_size=0.3 → split 70/30 según la práctica.
    """
    df = pd.read_csv(filepath)

    # Seleccionar columnas relevantes
    cols = BASE_COLS + ['Sex'] + SKILL_COLS + ['Position']
    df = df[cols].dropna()

    # One-hot encoding de Sex
    df = pd.get_dummies(df, columns=['Sex'], drop_first=False)
    sex_cols = [c for c in df.columns if c.startswith('Sex_')]

    # Feature columns
    feature_cols = BASE_COLS + sex_cols + SKILL_COLS

    X = df[feature_cols].values.astype(np.float32)
    y_raw = df['Position'].values

    # Codificar etiquetas
    le = LabelEncoder()
    y = le.fit_transform(y_raw).astype(np.int64)

    print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")
    print(f"Clases: {le.classes_} → {list(range(len(le.classes_)))}")
    print(f"Distribución: { {k: int((y==i).sum()) for i,k in enumerate(le.classes_)} }")

    # Normalización (CRUCIAL para convergencia)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train/Test split 70/30
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print(f"\nSplit {int((1-test_size)*100)}/{int(test_size*100)} → "
          f"Train: {len(X_train)}, Test: {len(X_test)}")

    # Convertir a tensores
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t  = torch.tensor(y_test,  dtype=torch.long)

    return X_train_t, X_test_t, y_train_t, y_test_t, le, X.shape[1]

In [11]:
X_train, X_test, y_train, y_test, label_encoder, n_features = load_and_preprocess(DATA_PATH)

train_iter = load_array((X_train, y_train), BATCH_SIZE, True)
test_iter  = load_array((X_test, y_test), BATCH_SIZE, False)

Dataset: 13921 muestras, 37 features
Clases: ['DEF' 'FWD' 'GK' 'MID'] → [0, 1, 2, 3]
Distribución: {'DEF': 4585, 'FWD': 2715, 'GK': 1550, 'MID': 5071}

Split 80/20 → Train: 11136, Test: 2785


In [12]:
# ─────────────────────────────────────────────────────────
# FUNCIÓN PARA CREAR MODELOS CONFIGURABLES
# ─────────────────────────────────────────────────────────

In [13]:
def build_model(
    input_size,
    hidden_sizes,          # lista: e.g. [64, 64] → 2 capas ocultas de 64 neuronas
    output_size=4,
    activation='relu',     # 'relu', 'tanh', 'leakyrelu', 'elu'
    dropout_rate=0.0,      # 0.0 = sin dropout
    batch_norm=False       # True = BatchNorm entre capas
):
    """
    Construye un MLP configurable con nn.Sequential.

    CORRECCIÓN: se usa una función (lambda) por activación para crear
    instancias nuevas de nn.Module en cada capa. Reusar el mismo objeto
    causa errores con BatchNorm y nn.Sequential.
    """
    # FIX: cada entrada es una función que crea una instancia nueva
    act_map = {
        'relu':      lambda: nn.ReLU(),
        'tanh':      lambda: nn.Tanh(),
        'leakyrelu': lambda: nn.LeakyReLU(0.1),
        'elu':       lambda: nn.ELU()
    }
    get_act = act_map.get(activation.lower(), lambda: nn.ReLU())

    layers = []
    prev_size = input_size

    for h in hidden_sizes:
        layers.append(nn.Linear(prev_size, h))
        if batch_norm:
            layers.append(nn.BatchNorm1d(h))
        layers.append(get_act())   # FIX: instancia nueva por capa
        if dropout_rate > 0.0:
            layers.append(nn.Dropout(dropout_rate))
        prev_size = h

    layers.append(nn.Linear(prev_size, output_size))
    return nn.Sequential(*layers)

In [14]:
# ─────────────────────────────────────────────────────────
# FUNCIÓN DE ENTRENAMIENTO CON TENSORBOARD
# ─────────────────────────────────────────────────────────

In [15]:
def train(net, train_iter, test_iter, writer, model_name,
          num_epochs=500, lr=0.001, print_every=10):

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    history = {
        'train_loss': [],
        'test_loss':  [],
        'train_acc':  [],
        'test_acc':   []
    }

    best_test_acc = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_epoch_ch3(net, train_iter, loss_fn, optimizer)
        test_loss  = evaluate_loss(net, test_iter, loss_fn)
        test_acc   = accuracy(net, test_iter)

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        writer.add_scalars(f'{model_name}/Loss',
                           {'Train': train_loss, 'Test': test_loss}, epoch)
        writer.add_scalars(f'{model_name}/Accuracy',
                           {'Train': train_acc,  'Test': test_acc},  epoch)

        if test_acc > best_test_acc:
            best_test_acc = test_acc

        if epoch % print_every == 0 or epoch == 1:
            print(f"[{model_name}] Epoch {epoch}/{num_epochs} | "
                  f"TrainLoss={train_loss:.4f} | TestLoss={test_loss:.4f} | "
                  f"TrainAcc={train_acc:.4f} | TestAcc={test_acc:.4f}")

    writer.flush()   # FIX: asegura que TensorBoard lea todos los datos
    print(f"✓ Mejor Accuracy Test [{model_name}]: {best_test_acc:.4f}")
    return best_test_acc, history

In [16]:
def plot_history(history, title):
    writer = SummaryWriter(log_dir=f'runs/{title}')
    
    for epoch, (trl, tel, tra, tea) in enumerate(zip(
        history['train_loss'],
        history['test_loss'],
        history['train_acc'],
        history['test_acc']
    ), start=1):
        writer.add_scalars(f'{title}/Loss',     {'Train': trl, 'Test': tel}, epoch)
        writer.add_scalars(f'{title}/Accuracy', {'Train': tra, 'Test': tea}, epoch)
    
    writer.close()
    print(f"✓ '{title}' registrado en runs/{title}")

# Lanzar TensorBoard inline en Jupyter
%load_ext tensorboard
%tensorboard --logdir runs

In [17]:
print("="*60)
print("FASE 1: COMPARACIÓN DE MODELOS BASE")
print("="*60)
os.makedirs("runs", exist_ok=True)

base_configs = {
    'chico':  {'hidden_sizes': [4,   4]},
    'medio':  {'hidden_sizes': [16, 16]},
    'grande': {'hidden_sizes': [256,256]}
}

base_results   = {}
base_histories = {}

for name, cfg in base_configs.items():
    print(f"\n>>> Entrenando modelo {name.upper()} <<<")
    net = build_model(n_features, activation='relu', **cfg)
    acc, history = train(net, train_iter, test_iter,
                         model_name=name,
                         num_epochs=200,
                         lr=0.001)
    base_results[name]   = acc
    base_histories[name] = history
    plot_history(history, name)   # abre su propio writer internamente

print("\n── Resultados Fase 1 ──")
for k, v in base_results.items():
    print(f"  {k:8s} → Acc Test: {v:.4f}")

FASE 1: COMPARACIÓN DE MODELOS BASE

>>> Entrenando modelo CHICO <<<


TypeError: train() missing 1 required positional argument: 'writer'

In [ ]:
print("RESULTADOS MODELOS BASE")
print("-"*40)

for k,v in base_results.items():
    print(f"{k.upper():10s} --> Accuracy Test: {v:.4f}")